# 第4章 · 分组统计与综合应用

本单元可以单独打开并从头运行，不依赖其他 Notebook 的变量或输出文件。全部小数据为本课程自编合成数据，不代表真实学生、订单或股票行情。

**学习方法**：先说明每行含义与预期结果，再运行代码；练习答案位于折叠单元格。使用课程 `.venv`，无需下载数据或额外安装依赖。

**阅读参考**：[Python for Data Analysis 对应章节](https://wesmckinney.com/book/data-aggregation)。本单元文字、数据与练习为课程自行编写。

### 1. 校园饮品订单：一行一笔订单

六笔合成订单用于手工核对。金额为数量乘单价；本例价格均已匹配，日期是实际有订单的日期。

In [1]:
import pandas as pd
sales = pd.DataFrame({
    "订单": [1, 2, 3, 4, 5, 6],
    "日期": ["2026-09-01"] * 3 + ["2026-09-02"] * 3,
    "门店": ["东", "东", "西", "东", "西", "西"],
    "商品": ["茶", "咖啡", "茶", "茶", "咖啡", "茶"],
    "数量": [2, 1, 1, 3, 2, 2], "单价": [10, 20, 10, 10, 20, 10]})
sales["日期"] = pd.to_datetime(sales["日期"])
sales["金额"] = sales["数量"] * sales["单价"]
sales

,订单,日期,门店,商品,数量,单价,金额
0,1,2026-09-01,东,茶,2,10,20
1,2,2026-09-01,东,咖啡,1,20,20
2,3,2026-09-01,西,茶,1,10,10
3,4,2026-09-02,东,茶,3,10,30
4,5,2026-09-02,西,咖啡,2,20,40
5,6,2026-09-02,西,茶,2,10,20


### 2. groupby：拆分、计算、合并

先按门店拆成小表，在各小表内求和，再组成汇总表。as_index=False 使门店保留为普通列，便于继续合并。

In [2]:
store_total = sales.groupby("门店", as_index=False)["金额"].sum()
print(store_total)
print(sales.groupby(["门店", "商品"])["金额"].sum())

  门店  金额
0  东  70
1  西  70
门店  商品
东   咖啡    20
    茶     50
西   咖啡    40
    茶     30
Name: 金额, dtype: int64


### 3. 命名聚合：明确每个指标的含义

总金额、订单数和平均每单金额是不同指标。命名聚合直接给结果列起名；金额均值的分母是订单行数。

In [3]:
summary = sales.groupby("门店", as_index=False).agg(
    总金额=("金额", "sum"),
    订单数=("订单", "size"),
    总杯数=("数量", "sum"),
    平均每单金额=("金额", "mean"))
print(summary)

  门店  总金额  订单数  总杯数     平均每单金额
0  东   70    3    6  23.333333
1  西   70    3    5  23.333333


### 4. size 与 count：分母不能混淆

size 统计组内行数，count 统计某列非缺失值数。先明确要数订单、有效金额还是不同顾客，再选择统计方法。

In [4]:
with_missing = sales.copy()
with_missing.loc[0, "金额"] = float("nan")
counts = with_missing.groupby("门店").agg(
    行数=("金额", "size"), 有效金额数=("金额", "count"))
print(counts)

    行数  有效金额数
门店           
东    3      2
西    3      3


**先动手**：东店的行数与有效金额数各是多少？金额缺失能否当作免费订单？

In [5]:
# 在这里完成练习；参考答案在下一单元格。

<details><summary>参考答案与检查</summary>

```python
print(counts.loc["东"].tolist())  # [3, 2]
# 不能：缺失表示不知道金额，免费需要明确记录为0。
```
</details>

### 5. 加权均值：平均每杯价格

平均每单金额不等于平均每杯价格。后者用总金额除以总杯数，相当于按购买数量给单价加权。

In [6]:
summary["平均每杯价格"] = summary["总金额"] / summary["总杯数"]
print(summary[["门店", "平均每单金额", "平均每杯价格"]])
print("全体每杯均价：", sales["金额"].sum() / sales["数量"].sum())

  门店     平均每单金额     平均每杯价格
0  东  23.333333  11.666667
1  西  23.333333  14.000000
全体每杯均价： 12.727272727272727


### 6. transform：把组内统计带回原行

agg 每组输出一行，transform 返回与原表等长且对齐的结果。用每笔金额除以所属门店总金额得到组内占比。

In [7]:
sales["门店总金额"] = sales.groupby("门店")["金额"].transform("sum")
sales["门店内占比"] = sales["金额"] / sales["门店总金额"]
print(sales[["订单", "门店", "金额", "门店内占比"]])
print(sales.groupby("门店")["门店内占比"].sum())

   订单 门店  金额     门店内占比
0   1  东  20  0.285714
1   2  东  20  0.285714
2   3  西  10  0.142857
3   4  东  30  0.428571
4   5  西  40  0.571429
5   6  西  20  0.285714
门店
东    1.0
西    1.0
Name: 门店内占比, dtype: float64


### 7. 排名与组内前两名

先按金额降序，再按订单号升序决定并列次序，组内取前两行。结果是两笔订单，而不一定是两个不同金额档位。

In [8]:
ordered = sales.sort_values(["门店", "金额", "订单"],
                             ascending=[True, False, True])
top2 = ordered.groupby("门店").head(2)
print(top2[["订单", "门店", "金额"]])
sales["组内名次"] = sales.groupby("门店")["金额"].rank(
    ascending=False, method="min")

   订单 门店  金额
3   4  东  30
0   1  东  20
4   5  西  40
5   6  西  20


### 8. 交叉表：计数与金额表不同

crosstab 默认数记录。pivot_table 显式指定金额与 sum 才是金额汇总。本例已知交易明细完整，因此没有订单的组合可填零。

In [9]:
print(pd.crosstab(sales["门店"], sales["商品"]))
amount_table = sales.pivot_table(index="门店", columns="商品",
    values="金额", aggfunc="sum", fill_value=0)
print(amount_table)

商品  咖啡  茶
门店       
东    1  2
西    1  2
商品  咖啡   茶
门店        
东   20  50
西   40  30


### 9. 按日汇总与滚动窗口

先汇总到“日”，再滚动才是相邻观测日的窗口。rolling(2) 是两行，不自动补齐日历中没有记录的日期。

In [10]:
daily = sales.groupby("日期")["金额"].sum().sort_index()
print(daily)
print(daily.rolling(2, min_periods=2).mean())
print(sales.set_index("日期")["金额"].resample("D").sum(min_count=1))

日期
2026-09-01    50
2026-09-02    90
Name: 金额, dtype: int64
日期
2026-09-01     NaN
2026-09-02    70.0
Name: 金额, dtype: float64
日期
2026-09-01    50
2026-09-02    90
Freq: D, Name: 金额, dtype: int64


### 10. 金融迁移：按股票计算收益率

合成长表每行一只股票一天的收盘价。先按股票和日期排序，在每只股票内部错位，避免把另一只股票的价格当作昨日价。

In [11]:
quotes = pd.DataFrame({
    "股票": ["A", "B", "A", "B", "A", "B"],
    "日期": pd.to_datetime(["2026-09-01"] * 2 +
                          ["2026-09-02"] * 2 + ["2026-09-03"] * 2),
    "收盘价": [100, 50, 102, 49, 101, 50]})
quotes = quotes.sort_values(["股票", "日期"])
quotes["昨日价"] = quotes.groupby("股票")["收盘价"].shift(1)
quotes["收益率"] = quotes["收盘价"] / quotes["昨日价"] - 1
print(quotes)

  股票         日期  收盘价    昨日价       收益率
0  A 2026-09-01  100    NaN       NaN
2  A 2026-09-02  102  100.0  0.020000
4  A 2026-09-03  101  102.0 -0.009804
1  B 2026-09-01   50    NaN       NaN
3  B 2026-09-02   49   50.0 -0.020000
5  B 2026-09-03   50   49.0  0.020408


### 11. 金融迁移：从长表到收益率矩阵

pivot 后每列一只股票；pct_change(fill_method=None) 不自动填补缺失价格。比较两种写法的同一观测值。

In [12]:
price_matrix = quotes.pivot(index="日期", columns="股票", values="收盘价")
return_matrix = price_matrix.pct_change(fill_method=None)
print(return_matrix)
expected = quotes.loc[(quotes["股票"] == "A") &
    (quotes["日期"] == "2026-09-02"), "收益率"].iloc[0]
assert abs(return_matrix.loc["2026-09-02", "A"] - expected) < 1e-12

股票                 A         B
日期                            
2026-09-01       NaN       NaN
2026-09-02  0.020000 -0.020000
2026-09-03 -0.009804  0.020408


### 12. 综合练习与验收

先用订单手算，再检查汇总表。全表金额应为 140 元、总杯数为 11；两门店各 70 元，但每杯均价不同。

In [13]:
assert sales["金额"].sum() == 140
assert sales["数量"].sum() == 11
assert summary["总金额"].sum() == sales["金额"].sum()
print(summary)

  门店  总金额  订单数  总杯数     平均每单金额     平均每杯价格
0  东   70    3    6  23.333333  11.666667
1  西   70    3    5  23.333333  14.000000


**先动手**：按“日期—门店”统计金额与杯数，并计算每杯均价。哪一天全体销售金额更高？

In [14]:
# 在这里完成练习；参考答案在下一单元格。

<details><summary>参考答案与检查</summary>

```python
report = sales.groupby(["日期", "门店"], as_index=False).agg(
    金额=("金额", "sum"), 杯数=("数量", "sum"))
report["每杯均价"] = report["金额"] / report["杯数"]
print(report)
print(daily.idxmax(), daily.max())  # 2026-09-02，90元
```
</details>

### 13. 综合案例起点：从原始订单到日报

下面重新开始一套独立的合成数据。每行原本应是一笔订单，但有重复登记、数量缺失和未知商品；目标是输出可核对的已知金额日报。

In [15]:
raw_orders = pd.DataFrame({
    "订单": ["01", "02", "03", "04", "05", "06", "01"],
    "日期": ["2026-09-01"] * 3 + ["2026-09-02"] * 3 + ["2026-09-01"],
    "门店": ["东", "东", "西", "东", "西", "西", "东"],
    "商品": [" t ", "C", "T", "T", "C", "X", " t "],
    "数量": ["2", "1", "缺失", "3", "2", "1", "2"]})
catalog = pd.DataFrame({"商品": ["T", "C"], "单价": [10, 20]})
raw_orders

,订单,日期,门店,商品,数量
0,01,2026-09-01,东,t,2
1,02,2026-09-01,东,C,1
2,03,2026-09-01,西,T,缺失
3,04,2026-09-02,东,T,3
4,05,2026-09-02,西,C,2
5,06,2026-09-02,西,X,1
6,01,2026-09-01,东,t,2


### 14. 清洗：原始值与统计值并存

保留 raw_orders。先统一商品代码，再删除完全相同的登记，转换日期和数量；原始数量列保留，便于回查缺失原因。

In [16]:
tidy_orders = raw_orders.copy()
tidy_orders["商品"] = tidy_orders["商品"].str.strip().str.upper()
tidy_orders = tidy_orders.drop_duplicates().reset_index(drop=True)
tidy_orders["日期"] = pd.to_datetime(tidy_orders["日期"])
tidy_orders["有效数量"] = pd.to_numeric(tidy_orders["数量"], errors="coerce")
assert tidy_orders["订单"].is_unique
print(tidy_orders)

   订单         日期 门店 商品  数量  有效数量
0  01 2026-09-01  东  T   2   2.0
1  02 2026-09-01  东  C   1   1.0
2  03 2026-09-01  西  T  缺失   NaN
3  04 2026-09-02  东  T   3   3.0
4  05 2026-09-02  西  C   2   2.0
5  06 2026-09-02  西  X   1   1.0


### 15. 合并：把待核订单单独列出

左连接保留每笔订单。已匹配商品且数量为正才进入本例金额统计；数量缺失和价格未知都保留在待核清单中。

In [17]:
enriched = tidy_orders.merge(catalog, on="商品", how="left",
    validate="many_to_one", indicator=True)
eligible = enriched["_merge"].eq("both") & enriched["有效数量"].gt(0)
pending = enriched.loc[~eligible].copy()
known = enriched.loc[eligible].copy()
known["金额"] = known["有效数量"] * known["单价"]
print(pending[["订单", "数量", "商品", "_merge"]])

   订单  数量 商品     _merge
2  03  缺失  T       both
5  06   1  X  left_only


### 16. 汇总：金额与覆盖范围一起报告

已知金额日报仅覆盖四笔订单，不能称为全部销售额。汇总前后金额应一致，同时报告两笔待核订单。

In [18]:
known_report = known.groupby(["日期", "门店"], as_index=False).agg(
    已知金额=("金额", "sum"), 可计算订单数=("订单", "size"))
print(known_report)
print("待核订单数：", len(pending))
assert len(raw_orders) == 7 and len(tidy_orders) == 6
assert len(known) == 4 and len(pending) == 2
assert known_report["已知金额"].sum() == 110

          日期 门店  已知金额  可计算订单数
0 2026-09-01  东  40.0       2
1 2026-09-02  东  30.0       1
2 2026-09-02  西  40.0       1
待核订单数： 2


### 17. 变形：缺失单元格不能随意填零

日报透视成“日期×门店”方便比较。西店首日有数量待核订单，不能把该格写成零销售；即使已有金额的格子也可能尚未完整。

In [19]:
known_wide = known_report.pivot(index="日期", columns="门店", values="已知金额")
pending_wide = pending.groupby(["日期", "门店"]).size().unstack("门店")
print("已知金额（不是完整销售额）：")
print(known_wide)
print("待核订单数：")
print(pending_wide)

已知金额（不是完整销售额）：
门店             东     西
日期                    
2026-09-01  40.0   NaN
2026-09-02  30.0  40.0
待核订单数：
门店          西
日期           
2026-09-01  1
2026-09-02  1


**先动手**：核实订单03数量为1、商品X是5元的水。重新清洗合并并汇总，完整金额应为125元。不要在最终报表中直接改金额。

In [20]:
# 在这里完成练习；参考答案在下一单元格。

<details><summary>参考答案与检查</summary>

```python
resolved_orders = tidy_orders.copy()
resolved_orders.loc[resolved_orders["订单"] == "03", "有效数量"] = 1
resolved_catalog = pd.concat([catalog, pd.DataFrame({
    "商品": ["X"], "单价": [5]})], ignore_index=True)
complete = resolved_orders.merge(resolved_catalog, on="商品", how="left",
                                validate="many_to_one", indicator=True)
assert complete["_merge"].eq("both").all()
complete["金额"] = complete["有效数量"] * complete["单价"]
final_report = complete.groupby(["日期", "门店"])["金额"].sum()
print(final_report)
assert final_report.sum() == 125
```
</details>